### EVALUATION

FAQ Dataset

In [1]:
from ingest import load_faq_data
documents = load_faq_data()

In [2]:
documents[10]

{'id': '316180784f',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: How many hours per week am I expected to spend on this course?',
 'answer': 'It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.'}

In [3]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

115

In [4]:
documents = documents_llm

In [5]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [6]:
doc

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

#### GENERATE QUESTIONS

Use structured output -> Tells the LLM to generate the output in some defined structure

In [7]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [8]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [9]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [10]:
import json

user_prompt = json.dumps(doc)

In [11]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [12]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [13]:
result = response.output_parsed

print(result)

questions=['I just found this course late — can I still jump in now?', 'Is it too late to join if I missed the start date?', 'Can I enroll after the course has already started?', 'If I join now, will I still be able to get the certificate?', 'Do I need to submit the project before submissions close to qualify for the certificate?']


In [14]:
print(result.questions)

['I just found this course late — can I still jump in now?', 'Is it too late to join if I missed the start date?', 'Can I enroll after the course has already started?', 'If I join now, will I still be able to get the certificate?', 'Do I need to submit the project before submissions close to qualify for the certificate?']


In [15]:
from evaluation_utils import llm_structured

In [16]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['I just found this course — is it still okay to join now?', 'Can new students still sign up late, or is it too late already?', 'If I join the course after it started, will I still be able to get a certificate?', "What do I need to do to be eligible for the certificate if I'm joining late?", 'Is there a deadline for submitting the project if I want the course certificate?']


In [17]:
usage.input_tokens, usage.output_tokens

(207, 96)

In [18]:
from evaluation_utils import calc_price

In [19]:
cost = calc_price(usage)

cost

{'input_cost': 0.00015525, 'output_cost': 0.000432, 'total_cost': 0.00058725}

In [20]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'I just found this course — is it still okay to join now?',
  'document': '74eb249bbf'},
 {'question': 'Can new students still sign up late, or is it too late already?',
  'document': '74eb249bbf'},
 {'question': 'If I join the course after it started, will I still be able to get a certificate?',
  'document': '74eb249bbf'},
 {'question': "What do I need to do to be eligible for the certificate if I'm joining late?",
  'document': '74eb249bbf'},
 {'question': 'Is there a deadline for submitting the project if I want the course certificate?',
  'document': '74eb249bbf'}]

In [21]:
from evaluation_utils import llm_structured_retry

In [22]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [23]:
import pandas as pd

In [24]:
pd.DataFrame(records)

,question,document
0,I just found this course — is it still okay to...,74eb249bbf
1,"Can new students still sign up late, or is it ...",74eb249bbf
2,"If I join the course after it started, will I ...",74eb249bbf
3,What do I need to do to be eligible for the ce...,74eb249bbf
4,Is there a deadline for submitting the project...,74eb249bbf


In [25]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [26]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/115 [00:00<?, ?it/s]

In [27]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

575

In [34]:
ground_truth[10]

{'question': 'How do students join the Office Hours or live workshop if the Zoom link isn’t public?',
 'document': '489dd1c9d9'}

In [35]:
usages[10]

ResponseUsage(input_tokens=253, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=93, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=346)

In [28]:
from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

0.08825025000000004

In [29]:
from evaluation_utils import calc_total_price

calc_total_price(usages)

0.08825025000000004

In [30]:
df_ground_truth = pd.DataFrame(ground_truth)

In [32]:
df_ground_truth.to_csv("data/ground_truth-new.csv", index=False)


In [33]:
df_ground_truth

,question,document
0,Can I still join the course if I found it late?,74eb249bbf
1,Am I allowed to enroll after the course has al...,74eb249bbf
2,"If I join now, can I still get a certificate?",74eb249bbf
3,What do I need to do to qualify for the certif...,74eb249bbf
4,"Is it too late to participate, or can I still ...",74eb249bbf
...,...,...
570,Why does pip keep giving me requests 2.28 when...,4b30b918bc
571,What command can I use to install requests str...,4b30b918bc
572,I’m getting a 401 Client Error while working w...,4b30b918bc
573,How do I force-install requests v2.32.3 from t...,4b30b918bc
